# YOLO26 E0 Seed 17

GitHub commit `504fa9dae6f1e4fc3f156a6ea93a6d9121b53515` is the canonical source. Google Drive is used only for the frozen Clean_v2 dataset, `yolo26s.pt`, and persistent experiment outputs. This notebook runs E0 only and never runs E1, E2, E3, or Test.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess

ASSETS = Path('/content/drive/MyDrive/Master_Drone_Detection_Assets')
DATASET_ASSET = ASSETS/'datasets/DUT_Anti_UAV_Clean_v2'
MODEL_ASSET = ASSETS/'models/yolo26s.pt'
OUTPUT_ASSET = ASSETS/'experiments/paper_yolo26'
REPO = Path('/content/Master_Drone_Detection')
assert DATASET_ASSET.is_dir(), DATASET_ASSET
assert MODEL_ASSET.is_file(), MODEL_ASSET
OUTPUT_ASSET.mkdir(parents=True, exist_ok=True)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', 'https://github.com/bassilarafat/Master_drone_detection.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '504fa9dae6f1e4fc3f156a6ea93a6d9121b53515'], cwd=REPO, check=True)
dataset_target = REPO/'02_datasets/DUT_Anti_UAV_Clean_v2'
model_target = REPO/'05_models/pretrained/paper/yolo26s.pt'
dataset_target.parent.mkdir(parents=True, exist_ok=True)
model_target.parent.mkdir(parents=True, exist_ok=True)
if dataset_target.exists() or dataset_target.is_symlink(): dataset_target.unlink() if dataset_target.is_symlink() else shutil.rmtree(dataset_target)
if model_target.exists() or model_target.is_symlink(): model_target.unlink()
dataset_target.symlink_to(DATASET_ASSET, target_is_directory=True)
model_target.symlink_to(MODEL_ASSET)
experiment_target = REPO/'04_experiments/paper_yolo26'
experiment_target.parent.mkdir(parents=True, exist_ok=True)
if experiment_target.exists() or experiment_target.is_symlink(): experiment_target.unlink() if experiment_target.is_symlink() else shutil.rmtree(experiment_target)
experiment_target.symlink_to(OUTPUT_ASSET, target_is_directory=True)
%cd /content/Master_Drone_Detection


In [ ]:
import subprocess, sys, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.157'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
import ultralytics
assert ultralytics.__version__ == '8.4.157', ultralytics.__version__
assert torch.cuda.is_available(), 'CUDA GPU is required'
print(torch.cuda.get_device_name(0), ultralytics.__version__)


In [ ]:
import hashlib, json
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
protocol = json.loads((REPO/'configs/paper/protocol_yolo26.json').read_text())
release = REPO/'configs/datasets/DUT_Anti_UAV_Clean_v2.release.json'
assert sha256(release) == protocol['dataset']['release_record_sha256']
assert sha256(MODEL_ASSET) == protocol['initialization']['sha256']
assert protocol['dataset']['release_id'].startswith('DUT_Anti_UAV_Clean_v2@')
assert protocol['controls']['test_locked'] is True
assert protocol['experiments']['E2_Y26_P2_random']['status'] == 'blocked'
assert protocol['experiments']['E3_Y26_P2_hard']['status'] == 'blocked'
print('Clean_v2 release, model SHA, Test lock, and E2/E3 blocks verified')


In [ ]:
import subprocess, sys, json
dry_run = REPO/'04_experiments/paper_yolo26/e0_dry_run.json'
subprocess.run([sys.executable, '-m', 'drone_detection.paper_cli', 'yolo26-dry-run', '--output', str(dry_run)], cwd=REPO, env={**__import__('os').environ, 'PYTHONPATH': 'src'}, check=True)
report = json.loads(dry_run.read_text())
assert report['status'] == 'READY_FOR_APPROVED_TRAINING'
assert report['experiment_id'] == 'E0_Y26' and report['seed'] == 17
assert report['epochs'] == 100 and report['batch'] == 1 and report['nbs'] == 16
assert report['optimizer'] == 'MuSGD' and report['test_locked'] is True
assert report['training_called'] is False
print('E0_Y26 dry-run PASS')


In [ ]:
preflight = REPO/'04_experiments/paper_yolo26/colab_preflight.json'
subprocess.run([sys.executable, '-m', 'drone_detection.paper_cli', 'yolo26-preflight', '--output', str(preflight)], cwd=REPO, env={**__import__('os').environ, 'PYTHONPATH': 'src'}, check=True)
pf = json.loads(preflight.read_text())
assert pf['status'] == 'PASS'
assert pf['models'][0]['strides'] == [8, 16, 32]
assert pf['training_called'] is False and pf['test_evaluated'] is False
print('YOLO26 preflight PASS; E0 strides [8, 16, 32] confirmed')


In [ ]:
# Final gate: this is the only cell that starts real training.
assert torch.cuda.is_available()
assert report['status'] == 'READY_FOR_APPROVED_TRAINING'
assert pf['status'] == 'PASS' and pf['models'][0]['strides'] == [8, 16, 32]
assert report['seed'] == 17 and report['epochs'] == 100
subprocess.run([sys.executable, '-m', 'drone_detection.paper_cli', 'yolo26-train-e0'], cwd=REPO, env={**__import__('os').environ, 'PYTHONPATH': 'src'}, check=True)
print('E0_Y26 Seed 17 completed or failed explicitly. Stop here for review; do not launch E1.')
